In [ ]:
"""
fitters_nb.ipynb

Tests the fitter object, meant to 
fit all models to a given session.

Author: Stellina X. Ao
Created: 2026-03-05
Last Modified: 2026-03-06
Python Version: 3.11
"""

from sg.fitter import LVMFamily

import shutup
import scienceplots  # noqa: F401
import matplotlib.pyplot as plt
import pickle
import numpy as np

%load_ext autoreload
%autoreload 2

# pretty plots
plt.style.use(["nature"])
plt.rcParams["figure.dpi"] = 200
%matplotlib widget
%config InlineBackend.print_figure_kwargs = {'bbox_inches':None}

# suppress warnings :-)
shutup.please()

## Fit one session with 4 additive latents

In [ ]:
trial_data_all = np.load(
    "../vars/trial_data_all_MM012_MM013_all5.npz", allow_pickle=True
)["arr_0"]
session_data_all = np.load(
    "../vars/session_data_all_MM012_MM013_all5.npz", allow_pickle=True
)["arr_0"]
unit_spike_times_all = np.load(
    "../vars/unit_spike_times_all_MM012_MM013_all5.npz", allow_pickle=True
)["arr_0"]
regions_all = np.load("../vars/regions_all_MM012_MM013_all5.npz", allow_pickle=True)[
    "arr_0"
]

subj_idx = 0
sess_idx = 5

unit_spike_times = unit_spike_times_all[subj_idx][sess_idx]
trial_data = trial_data_all[subj_idx][sess_idx]
session_data = session_data_all[subj_idx][sess_idx]
regions = regions_all[subj_idx][sess_idx]
trial_data["block_side"] = np.where(trial_data["block_side"] == "left", 1, -1)

In [ ]:
family = LVMFamily(
    trial_data=trial_data,
    spike_times=unit_spike_times,
    session_data=session_data,
    regions=regions,
    n_latents_mult=4,
    n_latents_addt=4,
)
family.fit_all()

In [ ]:
from squiggs.neuron_viewer import FitRenderer
from squiggs.renderers import NeuronViewer

r = FitRenderer(
    family.mod_ae_gain,  # sub for the model of interest
    x=family.test_dl.dataset[:],
    y=family.test_dl.dataset[:]["robs"].detach().cpu(),
)

nv = NeuronViewer(num_units=r.y.shape[1], render_func=r)

In [ ]:
from sg.eval_models import plot_summary

plot_summary(family, model=family.mod_offset, potato=family.strategy, mode="offset")

In [ ]:
with open("../vars/family.pkl", "wb") as f:
    pickle.dump(family, f)